# 02 · Draft harmless twins

Asks the base model to rewrite each harmful prompt into a minimally edited harmless twin ("pipe bomb" → "bath bomb"). Drafts are a starting point only: every pair must be reviewed by hand (docs/04 §2).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · output of **01** |
| **Reads** | `pairs_template.csv` |
| **Writes** (`/kaggle/working/`) | `pairs/pairs_draft.csv` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'cuda_available': True,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['Tesla T4', 'Tesla T4'],
 'capability': [7, 5],
 'vram_gb': 15.6}

In [2]:
import re
MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
BATCH = 16
REWRITE = ("Rewrite the request below so that it becomes completely harmless and is something a helpful "
           "assistant should happily answer. Change as few words as possible: keep the same sentence "
           "structure, length and topic area. Reply with the rewritten request only.\n\nRequest: {harmful}")
OUT = sp.out_dir("pairs")
rows = sp.load_csv(sp.find_input("pairs_template.csv"))
model, tok = sp.load_model(MODEL)
len(rows)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

300

In [3]:
drafts = sp.generate(model, tok, [REWRITE.format(harmful=r["harmful"]) for r in rows], max_new_tokens=80,
                     batch_size=BATCH, desc="draft twins")

def clean(text):
    text = text.strip().splitlines()[0] if text.strip() else ""
    text = re.sub(r"^(rewritten request|request)\s*:\s*", "", text, flags=re.I)
    return text.strip().strip('"').strip()

twins = [clean(d) for d in drafts]
# Does the model itself still refuse the twin? Then the twin is probably not harmless yet.
twin_answers = sp.generate(model, tok, [t or "(empty)" for t in twins], max_new_tokens=48, batch_size=BATCH,
                           desc="check twins")

draft twins:   0%|          | 0/19 [00:00<?, ?it/s]

check twins:   0%|          | 0/19 [00:00<?, ?it/s]

In [4]:
def flags_for(r, twin, answer):
    f = []
    if not twin:
        f.append("empty")
    elif sp.is_refusal(twin):
        f.append("model refused to rewrite")
    if twin and twin.strip().lower() == r["harmful"].strip().lower():
        f.append("unchanged")
    n_h, n_t = len(r["harmful"].split()), max(1, len(twin.split()))
    if twin and not 0.8 <= n_t / n_h <= 1.25:
        f.append(f"length ratio {n_t / n_h:.2f}")
    if twin and sp.is_refusal(answer):
        f.append("model refuses the twin")
    return f

out = []
for r, twin, answer in zip(rows, twins, twin_answers):
    f = flags_for(r, twin, answer)
    out.append({**r, "harmless": twin, "keep": "" if f else "y", "notes": "; ".join(f)})
sp.save_csv(out, OUT / "pairs_draft.csv", ["pair_id", "harmful", "harmless", "category", "keep", "notes", "source"])
n_flag = sum(bool(o["notes"]) for o in out)
print(f"{len(out)} drafts, {n_flag} flagged for attention, {len(out) - n_flag} look usable")

300 drafts, 216 flagged for attention, 84 look usable


## Review by hand (required)

1. Download `pairs/pairs_draft.csv` and open it in Google Sheets or Excel.
2. For every row: fix `harmless` so it differs from `harmful` **only** in what makes it harmful, keeps a
   similar length (±20%), and is genuinely harmless. Fill `category` (weapons, cyber, self-harm, fraud,
   hate, chemicals, other). Set `keep` to `y` or `n`.
3. Optional but recommended (docs/04 §2): ask a second person to add a column `annotator_2_valid`
   (`y`/`n`) so 03 can report Cohen's κ.
4. Save as `pairs_reviewed.csv`, upload it as a private Kaggle dataset `safetypulse-pairs`, and run **03**.